In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc 
import anndata as ad
import h5py 
import glob
import matplotlib.pyplot as plt
import os

In [ ]:
sc.settings.verbosity = 3             
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')
sc.set_figure_params(scanpy=True, figsize=(4,4))      

In [ ]:
base_path = '/home/cporter/atlas_remake_June_20_2025/'

# Annotate Epithelial Cells 

In [ ]:
# Load raw data for all cells
adata_raw = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_raw_withAnnotation.h5ad'))

In [ ]:
# Cell type variable 
cell_type = 'Epithelial'

In [ ]:
# Subcluster and annotate the cell subset
adata = adata_raw.copy()
del adata_raw 

adata = adata[adata.obs['Annotation_Tier1']==cell_type]
print(adata.shape)

In [ ]:
count_sum = adata.X.sum(axis=1)
# Normalize the data and store normalized data as its own layer for each normalization step 
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, inplace = True, target_sum=1e4)
adata.layers['norm_counts'] = adata.X.copy()
sc.pp.log1p(adata)
adata.layers['log_counts'] = adata.X.copy()

In [ ]:
# check that counts layer is actually integers 
print(adata.layers['counts'][0:20,0:20])
print(adata.layers['norm_counts'][0:20,0:20])
print(adata.layers['log_counts'][0:20,0:20])

# check that you get integers when you un-normalize the data - randomly checking index 3 
unlog1p = unlog1p = np.expm1(adata.X[3, :])
print(unlog1p)
count_check = unlog1p/10000*count_sum[3].item()
print(count_check)

In [ ]:
# Calculate and plot highly variable genes 
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pl.highly_variable_genes(adata)

# Store a copy of adata in the .raw field before subsetting to just variable genes (NOT RAW COUNTS)
adata.raw = adata

# Subset to just variable genes 
# Scale the data 
adata = adata[:, adata.var.highly_variable]
sc.pp.scale(adata, max_value=10)  

# Run PCA and generate PCA plots 
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca(adata, color='n_genes')
sc.pl.pca_variance_ratio(adata, log=True)

# Calculate nearest neighbors 
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

# Calculate
sc.tl.umap(adata)

# Calculate and plot leiden clusters 
sc.tl.leiden(adata, resolution=.1, key_added = 'leiden_res.1')
sc.pl.umap(adata, color=['leiden_res.1'], size=1)

adata.write_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_processed.h5ad'))

In [ ]:
# Optional jump in point 
adata = sc.read_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_processed.h5ad'))

In [ ]:
# Plot FRID
import random
colors = ["#%06x" % random.randint(0, 0xFFFFFF) for _ in range(192)]
sc.pl.umap(adata, color='FRID', size=3, palette=colors)

In [ ]:
# Integrate with harmony (patient batch) 
import scanpy.external as sce
sce.pp.harmony_integrate(adata, key="FRID")
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40, use_rep='X_pca_harmony')
sc.tl.umap(adata)

import random
colors = ["#%06x" % random.randint(0, 0xFFFFFF) for _ in range(192)]
sc.pl.umap(adata, color='FRID', size=5, palette=colors)

In [ ]:
# save object 
adata.write_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_harmony.h5ad'))

In [ ]:
# Optional jump in point 
adata = sc.read_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_harmony.h5ad'))

In [ ]:
# Calculate and visualize different clustering resolutions 
sc.tl.leiden(adata, resolution=.05, key_added='harmony_leiden_res.05')
sc.pl.umap(adata, color=['harmony_leiden_res.05'], size=4)
sc.tl.leiden(adata, resolution=.1, key_added='harmony_leiden_res.1')
sc.pl.umap(adata, color=['harmony_leiden_res.1'], size=4)
sc.tl.leiden(adata, resolution=.2, key_added='harmony_leiden_res.2')
sc.pl.umap(adata, color=['harmony_leiden_res.2'], size=4)
sc.tl.leiden(adata, resolution=.3, key_added='harmony_leiden_res.3')
sc.pl.umap(adata, color=['harmony_leiden_res.3'], size=4)
sc.tl.leiden(adata, resolution=.4, key_added='harmony_leiden_res.4')
sc.pl.umap(adata, color=['harmony_leiden_res.4'], size=4)
sc.tl.leiden(adata, resolution=.5, key_added='harmony_leiden_res.5')
sc.pl.umap(adata, color=['harmony_leiden_res.5'], size=4)
sc.tl.leiden(adata, resolution=.6, key_added='harmony_leiden_res.6')
sc.pl.umap(adata, color=['harmony_leiden_res.6'], size=4)
sc.tl.leiden(adata, resolution=.65, key_added='harmony_leiden_res.65')
sc.pl.umap(adata, color=['harmony_leiden_res.65'], size=4)
sc.tl.leiden(adata, resolution=.7, key_added='harmony_leiden_res.7')
sc.pl.umap(adata, color=['harmony_leiden_res.7'], size=4)
sc.tl.leiden(adata, resolution=.8, key_added='harmony_leiden_res.8')
sc.pl.umap(adata, color=['harmony_leiden_res.8'], size=4)
sc.tl.leiden(adata, resolution=1, key_added='harmony_leiden_res1')
sc.pl.umap(adata, color=['harmony_leiden_res1'], size=4)
sc.tl.leiden(adata, resolution=1.5, key_added='harmony_leiden_res1.5')
sc.pl.umap(adata, color=['harmony_leiden_res1.5'], size=4)
sc.tl.leiden(adata, resolution=2, key_added='harmony_leiden_res2')
sc.pl.umap(adata, color=['harmony_leiden_res2'], size=4)

In [ ]:
# Save harmony integrated object 
adata.write_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_harmony.h5ad'))

In [ ]:
# Run cell typist
import celltypist
from celltypist import models

#used as reference for comparing manual annotations based on sigantures
predictions = celltypist.annotate(
    adata.raw.to_adata(), 
    model = 'Cells_Intestinal_Tract.pkl', 
    majority_voting = True
)

predict = predictions.to_adata()
tmp = predict.obs['majority_voting'].astype("str")
adata.obs['majority_voting']=tmp
sc.pl.umap(
    adata, color='majority_voting', wspace=0.1, add_outline=False, size=6, 
    legend_fontsize=11, legend_fontoutline=1.5, frameon=False, palette='tab20')

In [ ]:
# Run cell typist
import celltypist
from celltypist import models

#used as reference for comparing manual annotations based on sigantures
predictions = celltypist.annotate(
    adata.raw.to_adata(), 
    model = 'Human_Colorectal_Cancer.pkl', 
    majority_voting = True
)

predict = predictions.to_adata()
tmp = predict.obs['majority_voting'].astype("str")
adata.obs['majority_voting']=tmp
sc.pl.umap(
    adata, color='majority_voting', wspace=0.1, add_outline=False, size=6, 
    legend_fontsize=11, legend_fontoutline=1.5, frameon=False, palette='tab20')

In [ ]:
# Optional jump in point 
adata = sc.read_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_processed_9-26-25_sanityChecks.h5ad'))

In [ ]:
# Calculate cluster marker genes 
sc.tl.rank_genes_groups(adata, "harmony_leiden_res.1", method="wilcoxon", use_raw = True)

# save marker genes 
df_all = pd.DataFrame()
clusters = np.unique(adata.obs['harmony_leiden_res.1'])
for i in clusters:
    
    df = sc.get.rank_genes_groups_df(adata, group = i)

    #order by zscore
    df['abs_logFC'] = np.absolute(df['logfoldchanges'])
    df = df[['names', 'scores', 'logfoldchanges', 'abs_logFC', 'pvals', 'pvals_adj']]
    print(i)
    print(df)
    
    i = i.replace(" / ", "_")
    i = i.replace(" ", "_")
    
    
    tmp=df['names'][0:100]
    df_all = pd.concat([df_all, tmp], axis=1)
df_all.columns=clusters
df_all.to_csv(os.path.join(base_path, f"results/{cell_type}_res.1_markerGenes_top100.csv"))

In [ ]:
df_all = pd.read_csv(os.path.join(base_path, f"results/{cell_type}_res.1_markerGenes_top100.csv"),index_col=0,)
# Look at top marker genes of each cluster 
for i in range(len(df_all.columns)):
    print(i)
    sc.pl.umap(adata, color=df_all[str(i)][0:40], ncols=10)

In [ ]:
# optional jump in point 
adata = sc.read_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_harmony.h5ad'))

In [ ]:
# Look at iCMS2 and iCMS3 scores 
df = pd.read_csv(os.path.join(base_path, 'docs/41588_2022_1100_MOESM3_ESM.csv'), keep_default_na=False)
subsets = df.columns
for s in subsets:
    sc.tl.score_genes(adata, df[s].values[0:df.shape[0]], score_name = s, use_raw=True)
    fig, ax = plt.subplots()
    sc.pl.umap(adata, color=s, use_raw=False, size=2, cmap='inferno', vmin='p15', ax=ax)
    fig.savefig(os.path.join(base_path, f'results/annotations_YOCRC_7-10-25/{s}_harmony_UMAP.pdf'), dpi=600, bbox_inches='tight')
    plt.close(fig)

In [ ]:
# Load original adata 
adata = sc.read_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_harmony.h5ad'))

In [ ]:
sc.pl.umap(adata, color=['CFTR', 'ANPEP', 'FOS', 'B2M', 'DUOX2', 'REG4', 'SPIB', 'LGR5'], size=3)

In [ ]:
# Check markers of other cell types
sc.pl.umap(adata, color=['PTPRC', 'EPCAM', 'PECAM1', 'CDH5', 'COL1A1', 'COL1A2', 'COL6A2', 'VWF'], size=3)

In [ ]:
# Cycling score 
cell_cycle_genes = [x.strip() for x in open(os.path.join(base_path, 'docs/regev_lab_cell_cycle_genes.txt'))]
s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]
sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)
sc.pl.umap(adata, color=['phase'])
sc.pl.violin(adata, 'phase', groupby='harmony_leiden_res.1')

In [ ]:
# Look at marker genes from polyp / CRC paper 
sc.pl.umap(adata, color=['LGR5', 'SMOC2', 'RGMB', 'PTPRO', 'EPHB2', 'LRIG1', 'BEST4', 'MUC2', 'RETNLB', 'FEV', 'RAB6B',
                        'SOX9', 'ASCL2', 'EPCAM', 'SMOC2', 'HNF4A', 'GPX2'])
sc.pl.umap(adata, color=['LRIG1', 'SOX9', 'RGMB', 'SMOC2', 'ASCL2', 'LGR5', 'EPHB2', 'ALCAM', 'CD44', 'EPCAM', 'PROM1'])

In [ ]:
# Run Kong et al epi signatures 
lk_markers = {
    'epithelial' : ['EPCAM', 'KRT8', 'KRT18'],
    'stromal' : ['CDH5', 'COL1A1', 'COL1A2', 'COL6A2', 'VWF'],
    'immune' : ['PTPRC', 'CD3D', 'CD3G', 'CD3E', 'CD79A', 'CD79B', 'CD14', 'CD68', 'CD83', 'CSF1R', 'FCER1G'],#'CD16', 
    'enterocytes' : ['RBP2', 'ANPEP', 'FABP2'],
    'stem_cells' : ['LGR5', 'ASCL2', 'SMOC2', 'RGMB', 'OLFM4'],
    'goblets' : ['CLCA1', 'SPDEF', 'FCGBP', 'ZG16', 'MUC2'],
    'paneth' : ['DEFA5', 'DEFA6', 'REG3A'],
    'tuft' : ['LRMP', 'SH2D6'],
    'enteroendocrine' : ['CHGA', 'CHGB', 'NEUROD1'],
    'cycling' : ['UBE2C', 'TOP2A', 'MKI67', 'HMGB2'],
}

for name, markers in lk_markers.items():
    sc.tl.score_genes(adata, markers, score_name = name, use_raw=True)
    sc.pl.umap(adata, color=markers, size=4)

sc.pl.umap(adata, color=list(lk_markers.keys()), size=4, cmap='inferno')

In [ ]:
lk_markers = {
    'Enterochromaffin' : ['CHGA', 'TPH1', 'CES1',  'SLC38A11', 'RAB3C'],
    'best4' : ['BEST4', 'CA7', 'CA4', 'SPIB', 'OTOP2'],#NOTCH2NL 
    'ca1_ca2_ca4neg' : ['CA1', 'SLC26A2', 'CA2', 'SLC26A3', 'KRT19', 'SELENBP1', 'PKIB', 'UGT2B17', 'CES2'], 
    'tmigd1_mep1a' : ['TMIGD1', 'MEP1A', 'APOA4', 'APOC3', 'APOA1', 'FABP6'],
    'tmigd1_mep1a_gsta1' : ['GSTA1', 'GSTA2', 'TMIGD1', 'MEP1A'], 
    'enteroendocrine' : ['PCSK1N', 'PYY', 'CHGA', 'GCG', 'CRYBA2', 'SCGN', 'FEV', 'SCG5', 'INSL5', 'MS4A8'], 
    'hbb_hba' : ['HBB', 'HBA2', 'HBA1'], 
    'mettl12_mafb' : ['MAFB'],#METTL12 
    'cycling' : ['UBE2C', 'PTTG1', 'HMGB2', 'TOP2A', 'CKS2', 'CENPW', 'CDKN3', 'STMN1', 'TUBB4B', 'HIST1H4C'], 
    'goblet_tff1neg' : [ 'MUC2', 'RETNLB', 'SPINK4', 'ITLN1', 'CLCA1', 'FCGBP', 'TFF3', 'ST6GALNAC1', 'LRRC26', 'REP15'], 
    'goblet_tff1pos' : ['MUC2', 'SPINK4', 'FCGBP', 'CLCA1', 'ZG16', 'TFF1', 'BCAS1', 'CEACAM5'], 
    'goblet_spink4' : ['SPINK4', 'MUC2', 'FCGBP', 'CLCA1', 'ITLN1', 'TFF3', 'TFF1', 'S100P', 'RETNLB', 'LRRC26'], 
    'l_cells' :  ['CHGA', 'NTS', 'PYY', 'GCG', 'CCK'], 
    'paneth' :  ['DEFA5', 'DEFA6', 'REG3A', 'PRSS1', 'ITLN2', 'PLA2G2A'], 
    'stem_olmfa' :  ['OLFM4', 'REG1A'], 
    'stem_olmfa_gsta1' :  ['FABP1', 'GSTA1', 'AKR1C3', 'KRT19', 'MAOA', 'CES2', 'CBR1', 'RBP2', 'PTGR1', 'LIMA1'], 
    'stem_olmfa_lgr5' :  ['LGR5', 'OLFM4'], 
    'stem_olmfa_pcna' :  ['PCNA', 'RANBP1', 'OLFM4', 'DUT', 'SIVA1'], #STRA13
    'tuft' :  ['SH2D6', 'LRMP', 'AVIL', 'BMX', 'AZGP1', 'MATK', 'TRPM5']} #7SK_ENSG00000260682
    
for name, markers in lk_markers.items():
    sc.tl.score_genes(adata, markers, score_name = name, use_raw=True)
    sc.pl.umap(adata, color=markers, size=4)

sc.pl.umap(adata, color=list(lk_markers.keys()), size=4, cmap='inferno')

In [ ]:
sc.pl.umap(adata, color=['pct_counts_mt'], size=4)

In [ ]:
# Load original adata 
adata = sc.read_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_harmony.h5ad'))

In [ ]:
sc.pl.umap(adata, color=['CHGA', 'CHGB', 'CACNA1A'], size=4)

In [ ]:
# Look at Pelka et al epi markers 
df = pd.read_csv(os.path.join(base_path, 'docs/crc_epi_signatures.csv'), keep_default_na=False)
subsets = df.columns
for s in subsets:
    sc.tl.score_genes(adata, df[s].values[1:df.shape[0]], score_name = s, use_raw=True)

sc.pl.umap(adata, color=subsets, size=2, cmap='inferno')

In [ ]:
# Look at Pelka et al malig programs 
df = pd.read_csv(os.path.join(base_path, 'docs/crc_malig_programs.csv'), keep_default_na=False)
subsets = df.columns
for s in subsets:
    sc.tl.score_genes(adata, df[s].values[1:df.shape[0]], score_name = s, use_raw=True)

sc.pl.umap(adata, color=subsets, size=2, cmap='inferno')

In [ ]:
# Load original adata 
adata = sc.read_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_harmony.h5ad'))

In [ ]:
# look at clusters 6 and 7 alone 
sc.pl.umap(adata, color=['harmony_leiden_res.1'], size=10, groups="6") # patient specific, but colonocyte-like 
sc.pl.umap(adata, color=['harmony_leiden_res.1'], size=10, groups="7") # stem-like 

In [ ]:
# Assign cluster annotations 
tier2annotation = {
    '0' : 'LGR5 stem cell-like', #
    '1' : 'CEACAM1 colonocyte-like', #
    '2' : 'MT-Ribo-hi epithelial', #
    '3' : 'MUC2 goblet-like', #
    '4' : 'Mixed - epithelial', #
    '5' : 'Patient-specific', #
    '6' : 'CEACAM1 colonocyte-like', 
    '7' : 'LGR5 stem cell-like'
}
adata.obs['Annotation_Tier2'] = adata.obs['harmony_leiden_res.1'].map(tier2annotation).astype('category')

adata.obs['Annotation_Tier2'] = adata.obs['Annotation_Tier2'].cat.add_categories("Enteroendocrine-like")
adata.obs['Annotation_Tier2'][adata.obs['harmony_leiden_res.8']=="11"]="Enteroendocrine-like"

In [ ]:
sc.pl.umap(adata, color=['Annotation_Tier2'], size=4)

In [ ]:
# save object 
adata.write_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_annotation.h5ad'))

In [ ]:
adata = sc.read_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_annotation.h5ad'))

In [ ]:
# Plot Tier2
colors = ['#000000', '#E69F00', '#0072B2', '#D55E00', '#009E73', '#56B4E9', '#CC79A7','#F0E442',]

fig, ax = plt.subplots()
sc.pl.umap(adata, color='harmony_leiden_res.1', size=1, ax=ax, show=True, palette=colors)
fig.savefig(os.path.join(base_path, 'results/annotations_EOCRC_DATE/LeidenClusters_Tier2_UMAP_Epithelial_Harmony.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
# Plot Tier2
colors = ['#000000', '#E69F00', '#0072B2', '#D55E00', '#009E73', '#56B4E9', '#CC79A7','#F0E442',]

fig, ax = plt.subplots()
sc.pl.umap(adata, color='Annotation_Tier2', size=1, ax=ax, show=True, palette=colors)
fig.savefig(os.path.join(base_path, 'results/annotations_EOCRC_DATE/Annotation_Tier2_UMAP_Epithelial_Harmony.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
# Plot FRID
import random
colors = ["#%06x" % random.randint(0, 0xFFFFFF) for _ in range(192)]

fig, ax = plt.subplots()
sc.pl.umap(adata, color='FRID', size=1, ax=ax, show=True, palette=colors)
fig.savefig(os.path.join(base_path, 'results/annotations_EOCRC_DATE/FRID_UMAP_EpiOnly_Harmony.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
adata_orig = sc.read_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_processed.h5ad'))

In [ ]:
# check that sizes check out 
print(sum(adata_orig.obs_names==adata.obs_names))
print(adata.shape)
print(adata_orig.shape)

In [ ]:
adata_orig.obs = adata.obs 

In [ ]:
adata_orig.write_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_annotation_noHarmony.h5ad'))

In [ ]:
# Plot Tier2
colors = ['#000000', '#E69F00', '#0072B2', '#D55E00', '#009E73', '#56B4E9', '#CC79A7','#F0E442',]

fig, ax = plt.subplots()
sc.pl.umap(adata_orig, color='Annotation_Tier2', size=1, ax=ax, show=True, palette=colors)
fig.savefig(os.path.join(base_path, 'results/annotations_EOCRC_DATE/Annotation_Tier2_UMAP_Epithelial_NoHarmony.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
# Plot and save metadata on UMAP
colors = ['#000000', '#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7']
sc.set_figure_params(figsize=(4, 4))

# Plot DECADE
fig, ax = plt.subplots()
sc.pl.umap(adata_orig, color='Decade', size=1, ax=ax, show=True, palette=colors)
fig.savefig(os.path.join(base_path, 'results/annotations_EOCRC_DATE/Decade_UMAP_EpiOnly_NoHarmony.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

# Plot AGE COHORT
fig, ax = plt.subplots()
sc.pl.umap(adata_orig, color='Cohort', size=1, ax=ax, show=True, palette=colors[5:7])
fig.savefig(os.path.join(base_path, 'results/annotations_EOCRC_DATE/Cohort_UMAP_EpiOnly_NoHarmony.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

# Plot FRID
import random
colors = ["#%06x" % random.randint(0, 0xFFFFFF) for _ in range(192)]

fig, ax = plt.subplots()
sc.pl.umap(adata_orig, color='FRID', size=1, ax=ax, show=True, palette=colors)
fig.savefig(os.path.join(base_path, 'results/annotations_EOCRC_DATE/FRID_UMAP_EpiOnly_NoHarmony.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)